# 0. Prépartaion de l'environement

In [1]:
# character Gras
B = '\033[1m'
BE = '\033[0m'

In [3]:
# Import des liberairies

from pyspark.sql.session import SparkSession

import pyspark.sql.functions as sf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from replace_accents import replace_accents_characters as replace_acc





In [5]:
#Chemin vers les données sur ma machine
CHEMIN = '../data/raw/autosurveillance-donnees-2022-12.csv'

#Création de la session spark
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# 1. Chargement et traitement préliminaire des données

In [6]:
# Data frame brute
df_brt = spark.read.format('csv').options(
                                    encoding = 'UTF-8'
                                    ,delimiter = ';'
                                    ,inferSchema = 'False'
                                    ,header = 'True'
                                    ).load(CHEMIN)
df_brt.printSchema()

root
 |-- Année: string (nullable = true)
 |-- Mois: string (nullable = true)
 |-- Finalité analyse: string (nullable = true)
 |-- Type_Code: string (nullable = true)
 |-- Type_Libellé: string (nullable = true)
 |-- Station_No Interne: string (nullable = true)
 |-- Station_Code SANDRE: string (nullable = true)
 |-- Station_Nom: string (nullable = true)
 |-- Departement_Numero: string (nullable = true)
 |-- Commune_Code INSEE: string (nullable = true)
 |-- Commune_Libellé: string (nullable = true)
 |-- XL93: string (nullable = true)
 |-- YL93: string (nullable = true)
 |-- Station_Capacite nominale: string (nullable = true)
 |-- Prelevement_Commentaire: string (nullable = true)
 |-- Point_Prelevement: string (nullable = true)
 |-- Prelevement_Date: string (nullable = true)
 |-- Parametre_Code SANDRE: string (nullable = true)
 |-- Parametre_Libelle: string (nullable = true)
 |-- Parametre_Type: string (nullable = true)
 |-- Support_Code: string (nullable = true)
 |-- Support_Libelle: str

In [8]:
# les alias des colonnes

# Renomage en se basant sur la convention SANDRE (le standard dans l'eau en France) _ 2 lettres de préfixe : 
# Cd = Code, Lb = Libellé, Dt = Date, Rs = Résultat, Mn = Mnémonique, St = Statut. 
# Nm = Nom, Année = An, Mois = Mo, No/Numéro = No, symbol = Sb, Type = Tp, point = pt

# Prelevement = Prel, Station, Stat, Departement = Dept, Support = Sup, Fraction = Frac,
# Parametre = Param, Unité = Unt, Analyse = Anly, Capacite = Cap,  Nominale = Nom


In [21]:
# Selectiions des colonnes utiles
df_ColSel= df_brt.select(
    #------Temporel---------#
    sf.col('Année').alias('AnPrel')
    ,sf.col('Mois').alias('MoPrel')
    ,sf.col('Prelevement_Date').alias('DtPrel')
    
    #----Localisation-------#
    ,sf.col('Departement_Numero').alias('NoDep')
    
    #-------Identification de la station ----#
    ,sf.col('Station_Code SANDRE').alias('CdSta')
    ,sf.concat_ws('__' , sf.col('Station_Nom'), sf.col('Station_No Interne')).alias('NmNoSta')
    ,sf.col('Type_Code').alias('CdTpSta')
    ,sf.col('Type_Libellé').alias('LbTpSta')

    #----- Caractéristiques de la station ------#
    ,sf.col('Station_Capacite nominale').alias('CapNomSta')
    
    #----Contexte du Prélevement-------#
    ,sf.col('Point_Prelevement').alias('CdPtPrel')
    ,sf.col('Support_Code').alias('CdSup')
    ,sf.col('Support_Libelle').alias('LbSup')
    ,sf.col('Fraction_Code').alias('CdFrac')
    ,sf.col('Fraction_Nom').alias('NmFrac')
    
    # ---- Paramètres mesuré ----#
    ,sf.col('Parametre_Code SANDRE').alias('CdPar')
    ,sf.col('Parametre_Libelle').alias('LbPar')

    #---- Resulats et mesures ---#
    ,sf.regexp_replace('Resultat_Valeur', ',' , '.').alias('RsAna')
    ,sf.col('Resultat_Unite_Symbole').alias('SbUnt')
    
)
print(f"Total lignes : {df_ColSel.count()}")
df_ColSel.printSchema()

df_ColSel.select('AnPrel','MoPrel','DtPrel','NoDep','CdSta','NmNoSta','CdTpSta','LbTpSta','CapNomSta').show(2,truncate=False)
df_ColSel.select('CdPtPrel','CdSup','LbSup','CdFrac','NmFrac','CdPar','LbPar','RsAna','SbUnt').show(2,truncate=False)


Total lignes : 300765
root
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = false)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- CdPtPrel: string (nullable = true)
 |-- CdSup: string (nullable = true)
 |-- LbSup: string (nullable = true)
 |-- CdFrac: string (nullable = true)
 |-- NmFrac: string (nullable = true)
 |-- CdPar: string (nullable = true)
 |-- LbPar: string (nullable = true)
 |-- RsAna: string (nullable = true)
 |-- SbUnt: string (nullable = true)

+------+------+----------+-----+------------+----------------------+-------+-------------------------+---------+
|AnPrel|MoPrel|DtPrel    |NoDep|CdSta       |NmNoSta               |CdTpSta|LbTpSta                  |CapNomSta|
+------+------+----------+-----+------------+----------------

In [107]:
# liste de validation de la structure de la données 
regles_Validation = {
     #------ temporel ----
    'AnPrel' : r'^\d{4}$'
    ,'MoPrel': r'^\d{2}$'
    ,'DtPrel': r'^\d{2}/\d{2}/\d{4}$'
        
    #------ localisation ----
    ,'NoDep': r'^\d{2,3}$' 
    
     #------ identification station ----
    ,'CdSta': r'^\d+[A-Z]?\d*$'
    ,'NmNoSta': r'^.+$'   
    ,'CdTpSta': r'^\d+$'
    ,'LbTpSta': r'^.+$'
    
    # ---------Caractéristiques de la station
    ,'CapNomSta': r'^\d+$' 
    
    #------ contexte prélèvement ----
    ,'CdPtPrel': r'^[A-Z]\d+[a-z]?$' 
    ,'CdSup': r'^\d+$'
    ,'LbSup': r'.+'
    ,'CdFrac': r'^\d+$'
    ,'NmFrac': r'.+'

    #------ parametre mesuré ----
    ,'CdPar': r'^\d+$'
    ,'LbPar': r'.+'
    
    #------Résultat ----
    ,'RsAna': r'^\s*-?\d+([.,]\d+)?\s*$'
    ,'SbUnt': r'.+'
    
}

# Unit tests

# df_ColSel.select(sf.col('CdSta')).filter(sf.col('CdSta').rlike(r'^\d+[A-Z]\d*$')).show(5)
# df_ColSel.select(sf.col('NmNoSta')).filter(~sf.col('NmNoSta').rlike(r'^.+__\d*_\d+$')).show(5)
# df_ColSel.select(sf.col('CdTpSta')).filter(sf.col('CdTpSta').rlike(r'^\d{2}$')).show(5)

In [275]:
# Test de validation de la structure de la données
lignes_tot = df_ColSel.count()
print(f'Nombre de lignes totale: {lignes_tot}')

###----tableau de synthèse---------
#en-tete
print(f"{B} {'Colonne':<10} | {'Nb Nulls':>10} %{'Nulls':>10} | {'Nb Anomalies':>10} %{'Anomalies':>10} | {'Statute':<10} | {'Echantillon':<10} {BE}")
print(f"{B} {'--'*63} {BE}")

# Boucle unique : 1 passe par colonne pour calculer null + anomalies
for coln, pattern in regles_Validation.items():
    nulls = df_ColSel.filter((sf.col(coln).isNull())|(sf.col(coln)=="")).count()
    anomalies = df_ColSel.filter(~sf.col(coln).rlike(pattern)).count()
    pct_n = 100 * nulls/lignes_tot
    pct_a = 100 * anomalies/lignes_tot
    # Aperçu sur les anomalies
    
    valeurs_pouries = (
        df_ColSel
        .select(sf.col(coln))
        .filter(~sf.col(coln).rlike(pattern))
        .distinct()
        .limit(3)
        .collect()
    )
    if len(valeurs_pouries)==0:
        echontillon = 'aucune anomalie'
    else:
        echontillon =  [str(row[coln]) for row in valeurs_pouries]
    
    # condition d'affichage
    
    if pct_a == 0 and pct_n == 0:
        statut = 'Parfait'
    elif pct_a < 0.01 or pct_n < 0.1:
        statut = 'À Revoir'
    else:
        statut = 'Pourries'
    
    print(f" {coln:<10} | {nulls:>10} {pct_n:>10.2f}% | {anomalies:>10} {pct_a:>10.2f}% | {statut:<10} | {echontillon}\n")
    # print(f"{echontillon}")
print(f"{B} {'--'*63} {BE}") 


Nombre de lignes totale: 300765
 Colonne    |   Nb Nulls %     Nulls | Nb Anomalies % Anomalies | Statute    | Echantillon 
 ------------------------------------------------------------------------------------------------------------------------------ 
 AnPrel     |         15       0.00% |       2744       0.91% | À Revoir   | ['";"S11"', '";"A4"', 'Le prélèvement est le même qu\'en cas de by-pass."']

 MoPrel     |        824       0.27% |       1935       0.64% | Pourries   | ['03/12/2022', '18/12/2022', '14/12/2022']

 DtPrel     |       4431       1.47% |        130       0.04% | Pourries   | ['Correcte']

 NoDep      |        957       0.32% |        669       0.22% | Pourries   | ['Refus de degrillage brut', 'Eau brute', "Boue d'épuration"]

 CdSta      |        957       0.32% |        669       0.22% | Pourries   | ['Refus de dégrillage', 'Eau', 'Quantitatif']

 NmNoSta    |        957       0.32% |        957       0.32% | Pourries   | ['']

 CdTpSta    |        957       0.3

[]